In [ ]:
import numpy as np
from scipy.io.wavfile import write

# 1. TEMEL AYARLAR
ornekleme_hizi = 44100  # CD kalitesi (Sample Rate)
tempo = 120             # Dakikadaki vuruş sayısı (BPM) - Biraz ağır, hüzünlü olsun
birim_sure = 60 / tempo # Bir vuruşun saniye cinsinden süresi

# 2. HİCAZ MAKAMI FREKANSLARI (Hertz)
notalar = {
    "La4": 440.00,   # Dügah (Karar)
    "Si4_b": 466.16, # Dik Kürdi (Si Bakiye Bemol - Yaklaşık)
    "Do5_s": 554.37, # Hicaz (Do Bakiye Diyez - Yaklaşık)
    "Re5": 587.33,   # Neva
    "Mi5": 659.25,   # Hüseyni
    "Fa5_s": 739.99, # Eviç
    "Sol5": 783.99,  # Gerdaniye
    "Es": 0.0        # Sus
}

# 3. BESTEMİZİN NOTALARI (Nota Adı, Vuruş Süresi)
sarki = [
    # 1. Mısra: "Bu Şe-hir Be-ni Bo-ğu-yor"
    ("Sol5", 0.5), ("Fa5_s", 0.5), ("Sol5", 0.5), ("Fa5_s", 0.5), ("Mi5", 1.0), ("Re5", 2.0),

    # 2. Mısra: "Bil-mem Ba-na Ne O-lu-yor"
    ("Re5", 0.5), ("Mi5", 0.5), ("Re5", 0.5), ("Do5_s", 0.5), ("Si4_b", 0.5), ("Do5_s", 0.5), ("Re5", 2.0),

    # 3. Mısra: "Çö-ken Ka-ran-lı-ğın İ-çin-de"
    ("Re5", 0.5), ("Do5_s", 0.5), ("Si4_b", 0.5), ("La4", 0.5), ("Si4_b", 0.5), ("Do5_s", 0.5), ("Re5", 2.0),

    # 4. Mısra: "U-mut-la-rım Tü-ke-ni-yor"
    ("Do5_s", 0.5), ("Si4_b", 0.5), ("La4", 0.5), ("Es", 0.5), ("Sol5", 0.5), ("Fa5_s", 0.5), ("Sol5", 0.5), ("La4", 2.0)
]

# 4. SES SENTEZLEME FONKSİYONU (Sinüs Dalgası)
def ses_olustur(frekans, sure_sn):
    t = np.linspace(0, sure_sn, int(ornekleme_hizi * sure_sn), False)

    if frekans == 0: # Es (Sus) durumu
        dalga = np.zeros_like(t)
    else:
        # Saf Sinüs Dalgası Formülü: A * sin(2 * pi * f * t)
        dalga = 0.5 * np.sin(2 * np.pi * frekans * t)

        # "Click" sesini önlemek için yumuşak giriş-çıkış (Envelope)
        zarf_suresi = 0.05 # 50ms yumuşatma
        zarf_sample = int(ornekleme_hizi * zarf_suresi)
        if len(dalga) > 2 * zarf_sample:
            dalga[:zarf_sample] *= np.linspace(0, 1, zarf_sample)
            dalga[-zarf_sample:] *= np.linspace(1, 0, zarf_sample)

    return dalga

# 5. PARÇALARI BİRLEŞTİRME
tum_sesler = []

for nota_adi, vurus in sarki:
    frekans = notalar[nota_adi]
    sure = vurus * birim_sure
    tum_sesler.append(ses_olustur(frekans, sure))

# Listeyi tek bir diziye çevir
bilesik_ses = np.concatenate(tum_sesler)

# 16-bit WAV formatına uygun hale getir
bilesik_ses_int16 = np.int16(bilesik_ses * 32767)

# Dosyayı kaydet
write('hicaz_sarki.wav', ornekleme_hizi, bilesik_ses_int16)

print("Müjde azizim! 'hicaz_sarki.wav' dosyası hazırlandı.")